In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    if len(filenames) > 0:
        print("Files:", filenames[:5])
    print("-" * 50)

In [ ]:
TRAIN_PATH = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/train"
TEST_PATH = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/test"

In [ ]:
import pandas as pd
import os

train_path = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/train"

files = os.listdir(train_path)

horizontal_files = [f for f in files if "horizontal_well.csv" in f]

print("Number of horizontal wells:", len(horizontal_files))
print("First file:", horizontal_files[0])

sample = pd.read_csv(os.path.join(train_path, horizontal_files[0]))

print("\nColumns:")
print(sample.columns.tolist())

print("\nShape:")
print(sample.shape)

sample.head()

In [ ]:
import pandas as pd
import os

train_path = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/train"

sample_file = "b0f53bf4__horizontal_well.csv"

df = pd.read_csv(os.path.join(train_path, sample_file))

print(df.isnull().sum())

In [ ]:
import pandas as pd
import os
from tqdm import tqdm

train_path = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/train"

all_data = []

files = [f for f in os.listdir(train_path)
         if "__horizontal_well.csv" in f]

for file in tqdm(files):
    df = pd.read_csv(os.path.join(train_path, file))

    df["well_id"] = file.split("__")[0]

    all_data.append(df)

train_df = pd.concat(all_data, ignore_index=True)

print("Shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns.tolist())

In [ ]:
!pip install lightgbm -q

In [ ]:
sample_df = train_df.sample(300000, random_state=42)

print(sample_df.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import numpy as np

features = [
    'MD',
    'X',
    'Y',
    'Z',
    'ANCC',
    'ASTNU',
    'ASTNL',
    'EGFDU',
    'EGFDL',
    'BUDA',
    'GR',
    'TVT_input'
]

X = sample_df[features].copy()
y = sample_df['TVT']

X['GR'] = X['GR'].fillna(-999)
X['TVT_input'] = X['TVT_input'].fillna(-999)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=64,
    random_state=42
)

model.fit(X_train, y_train)

preds = model.predict(X_valid)

rmse = np.sqrt(mean_squared_error(y_valid, preds))

print("RMSE:", rmse)